In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import TargetEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, 
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    accuracy_score,
    f1_score
)
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CÁC CLASS XỬ LÝ (GIỮ NGUYÊN LOGIC)
# ==============================================================================

class LogicalCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X_out = X.copy()
        
        # --- CLEAN SLEEP DURATION ---
        if 'Sleep Duration' in X_out.columns:
            def clean_sleep(val):
                s = str(val).lower().strip()
                if any(x in s for x in ['1-2', '2-3', '3-4',  '1-3']): return 'Less than 4 hours'
                elif any(x in s for x in ['less than 5','4-5','than 5']): return '4-5 hours'
                elif any(x in s for x in ['5-6', '4-6', '3-6','than 5']): return '5-6 hours'
                elif any(x in s for x in ['7-8', '6-8', '6-7']): return '6-8 hours'
                elif any(x in s for x in ['more than 8', '8-9', '9-11', '10-11']): return 'More than 8 hours'
                else: return 'Unknown'
            X_out['Sleep Duration'] = X_out['Sleep Duration'].apply(clean_sleep)

        if 'Dietary Habits' in X_out.columns:
            def clean_diet(val):
                s = str(val).lower().strip()
                if s in ['healthy', 'more healthy']: return 'Healthy'
                elif s in ['moderate']: return 'Moderate'
                elif s in ['unhealthy', 'less than healthy', 'no healthy', 'less healthy']: return 'Unhealthy'
                else: return 'Unknown'
            X_out['Dietary Habits'] = X_out['Dietary Habits'].apply(clean_diet)

        if 'Profession' in X_out.columns and 'Degree' in X_out.columns:
            X_out['Occupation'] = X_out['Profession'].fillna(X_out['Degree'])
            
        if 'Work Pressure' in X_out.columns and 'Academic Pressure' in X_out.columns:
            X_out['Pressure'] = X_out['Work Pressure'].fillna(X_out['Academic Pressure'])

        if 'Job Satisfaction' in X_out.columns and 'Study Satisfaction' in X_out.columns:
            X_out['Satisfaction'] = X_out['Job Satisfaction'].fillna(X_out['Study Satisfaction'])
            
        return X_out

class RareLabelEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, variables=None, threshold=5):
        self.variables = variables or []
        self.threshold = threshold
        self.valid_labels_ = {} 

    def fit(self, X, y=None):
        for col in self.variables:
            if col in X.columns:
                counts = X[col].value_counts()
                self.valid_labels_[col] = counts[counts > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X_out = X.copy()
        for col in self.variables:
            if col in X_out.columns:
                valid_list = self.valid_labels_.get(col, [])
                X_out[col] = np.where(X_out[col].isin(valid_list), X_out[col], 'Other')
        return X_out

class ScoreBasedSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01): 
        self.threshold = threshold
        self.selected_features_ = []
    def fit(self, X, y):
        print(f"\n--- [SELECTOR] Calculating Mutual Information... ---")
        X_temp = X.copy()
        cat_cols = X_temp.select_dtypes(include=['object', 'category']).columns
        num_cols = X_temp.select_dtypes(exclude=['object', 'category']).columns
        X_temp[num_cols] = X_temp[num_cols].fillna(0)
        ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_temp[cat_cols] = ord_enc.fit_transform(X_temp[cat_cols].fillna('Missing'))
        mi_scores = mutual_info_classif(X_temp, y, discrete_features='auto', random_state=42)
        score_df = pd.DataFrame({'feature': X_temp.columns, 'score': mi_scores}).sort_values(by='score', ascending=False)
        print(score_df.head(10))
        self.selected_features_ = score_df[score_df['score'] > self.threshold]['feature'].tolist()
        print(f"Selected {len(self.selected_features_)} features.")
        return self
    def transform(self, X):
        return X[self.selected_features_]

# ==============================================================================
# 2. PROCESS FLOW
# ==============================================================================
print(">>> 1. LOADING DATA...")
try:
    df_train = pd.read_csv('train.csv').drop_duplicates()
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file 'train.csv'")
    exit()
target_col = 'Depression'
for c in ['id', 'Name', 'PassengerId']:
    if c in df_train.columns: df_train.drop(c, axis=1, inplace=True)

X = df_train.drop(target_col, axis=1)
y = df_train[target_col]

# --- CHIA TẬP TRAIN / VAL / TEST (Tỷ lệ 70% / 15% / 15%) ---
print("\n>>> 2. SPLITTING DATA (Train/Val/Test)...")
# B1: Tách 15% làm Test set
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

# B2: Tách phần còn lại (85%) thành Train và Val. 
# Để Val chiếm ~15% tổng số (giống Test), ta cần tách khoảng 17.65% của tập X_temp
# (0.15 / 0.85 approx 0.1765)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)

print(f"Train size: {X_train.shape[0]} | Val size: {X_val.shape[0]} | Test size: {X_test.shape[0]}")

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# --- PRE-PROCESSING PIPELINE ---
print("\n>>> 3. PRE-PROCESSING & FEATURE SELECTION...")
vars_to_rare = ['Occupation', 'Degree', 'Profession', 'City']

# Pipeline xử lý thô & gom nhóm
pre_cleaner = Pipeline([
    ('cleaner', LogicalCleaner()), 
    ('rare_encoder', RareLabelEncoder(variables=vars_to_rare, threshold=5))
])

# Fit trên Train, Transform cho cả Train, Val, Test
X_train_pre = pre_cleaner.fit_transform(X_train, y_train)
X_val_pre = pre_cleaner.transform(X_val)
X_test_pre = pre_cleaner.transform(X_test)

# Feature Selection (Chỉ học từ Train)
selector = ScoreBasedSelector(threshold=0.01) 
selector.fit(X_train_pre, y_train)
final_features = selector.selected_features_

X_train_selected = X_train_pre[final_features]
X_val_selected = X_val_pre[final_features]
X_test_selected = X_test_pre[final_features]

# --- TRANSFORMER CHO MODEL ---
print(f"\n>>> 4. PREPARING FOR MODEL ({len(final_features)} features)...")
cat_cols = X_train_selected.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train_selected.select_dtypes(exclude=['object', 'category']).columns.tolist()

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Other')),
    ('target_enc', TargetEncoder(smooth='auto', random_state=42)),
    ('scaler', StandardScaler())
])

final_preprocessor = ColumnTransformer(
    transformers=[('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)],
    verbose_feature_names_out=False
)

# Fit trên Train
X_train_processed = final_preprocessor.fit_transform(X_train_selected, y_train)
# Transform Val và Test
X_val_processed = final_preprocessor.transform(X_val_selected)
X_test_processed = final_preprocessor.transform(X_test_selected)

# --- TRAINING ---
print("\n>>> 5. TRAINING XGBOOST...")
xgb_model = xgb.XGBClassifier(
    n_estimators=3000, learning_rate=0.01, max_depth=5,
    scale_pos_weight=scale_pos_weight, eval_metric='aucpr',
    early_stopping_rounds=50, random_state=42, n_jobs=-1
)

xgb_model.fit(
    X_train_processed, y_train,
    eval_set=[(X_train_processed, y_train), (X_val_processed, y_val)],
    verbose=100
)

# --- FINDING OPTIMAL THRESHOLD (ON VALIDATION SET) ---
print("\n>>> 6. OPTIMIZING THRESHOLD...")
y_val_prob = xgb_model.predict_proba(X_val_processed)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal Threshold (from Val set): {best_threshold:.4f}")

# --- EVALUATION (ON TEST SET) ---
print("\n" + "="*40)
print(">>> 7. FINAL EVALUATION ON TEST SET")
print("="*40)

y_test_prob = xgb_model.predict_proba(X_test_processed)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int)

# Các chỉ số đánh giá
auc_score = roc_auc_score(y_test, y_test_prob)
acc_score = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
conf_matrix = confusion_matrix(y_test, y_test_pred)

print(f"ROC-AUC Score: {auc_score:.4f}")
print(f"Accuracy:      {acc_score:.4f}")
print(f"F1-Score:      {f1:.4f}")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_test_pred))

print("\n--- Confusion Matrix ---")
print(conf_matrix)
print("(TN  FP)")
print("(FN  TP)")

# ==============================================================================
# EXPORT CONFIG & MODEL
# ==============================================================================
print("\n>>> 8. EXPORTING SYSTEM...")
valid_labels_dict = pre_cleaner.named_steps['rare_encoder'].valid_labels_

ui_config = {
    "model_threshold": float(best_threshold),
    "input_fields": []
}

ui_features_set = set()
ui_features_set.add("Working Professional or Student")

BLACKLIST_UI = ['Occupation', 'Pressure', 'Satisfaction']

for feat in final_features:
    if feat == 'Occupation':
        ui_features_set.add('Degree')
        ui_features_set.add('Profession')
    elif feat == 'Pressure':
        ui_features_set.add('Academic Pressure')
        ui_features_set.add('Work Pressure')
    elif feat == 'Satisfaction':
        ui_features_set.add('Study Satisfaction')
        ui_features_set.add('Job Satisfaction')
    elif feat not in BLACKLIST_UI:
        ui_features_set.add(feat)

for col in ui_features_set:
    if col == "Working Professional or Student": continue
    if col not in X_train.columns: continue

    field_info = {"name": col, "label": col.replace("_", " ").title()}
    is_numeric = pd.api.types.is_numeric_dtype(X_train[col].dtype)
    is_scale_var = any(x in col for x in ['Pressure', 'Satisfaction', 'Stress'])
    
    if is_numeric and not is_scale_var:
        field_info["type"] = "number"
        field_info["min"] = float(X_train[col].min())
        field_info["max"] = float(X_train[col].max())
    else:
        field_info["type"] = "select"
        
        # Logic lấy options
        if col in ['Sleep Duration', 'Dietary Habits']: 
            temp_cleaner = LogicalCleaner()
            temp_df = temp_cleaner.transform(X_train[[col]])
            raw = temp_df[col].unique().tolist()
        elif col in valid_labels_dict:
            raw = valid_labels_dict[col]
        else:
            raw = X_train[col].value_counts().head(30).index.tolist()

        options = [str(x) for x in raw if str(x) != 'nan' and str(x) != 'Unknown']
        options = sorted(options)
        
        if col in ['Degree', 'Profession']:
            if 'Other' not in options: options.append('Other')
            
        field_info["options"] = options

    ui_config["input_fields"].append(field_info)

with open('model_ui_config.json', 'w', encoding='utf-8') as f:
    json.dump(ui_config, f, indent=4, ensure_ascii=False)

# Pipeline dùng cho inference (đã bao gồm các bước xử lý)
pipeline_inference = Pipeline([
    ('cleaner', pre_cleaner.named_steps['cleaner']),
    ('rare_encoder', pre_cleaner.named_steps['rare_encoder'])
])

full_system = {
    'selector_pipeline': pipeline_inference, 
    'preprocessor': final_preprocessor,
    'model': xgb_model,
    'threshold': best_threshold,
    'required_features': final_features 
}
joblib.dump(full_system, 'depression_prediction_system.pkl')
print("\n✅ System Saved Successfully.")

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import TargetEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, 
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    accuracy_score,
    f1_score
)
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings('ignore')


In [5]:
class LogicalCleaner(BaseEstimator, TransformerMixin):
    """Xử lý dữ liệu thô: clean Sleep, Dietary, gộp cột"""
    def fit(self, X, y=None): return self
    def transform(self, X):
        X_out = X.copy()
        # Clean Sleep Duration
        if 'Sleep Duration' in X_out.columns:
            def clean_sleep(val):
                s = str(val).lower().strip()
                if any(x in s for x in ['1-2', '2-3', '3-4',  '1-3']): return 'Less than 4 hours'
                elif any(x in s for x in ['less than 5','4-5','than 5']): return '4-5 hours'
                elif any(x in s for x in ['5-6', '4-6', '3-6','than 5']): return '5-6 hours'
                elif any(x in s for x in ['7-8', '6-8', '6-7']): return '6-8 hours'
                elif any(x in s for x in ['more than 8', '8-9', '9-11', '10-11']): return 'More than 8 hours'
                else: return 'Unknown'
            X_out['Sleep Duration'] = X_out['Sleep Duration'].apply(clean_sleep)

        # Clean Dietary Habits
        if 'Dietary Habits' in X_out.columns:
            def clean_diet(val):
                s = str(val).lower().strip()
                if s in ['healthy', 'more healthy']: return 'Healthy'
                elif s in ['moderate']: return 'Moderate'
                elif s in ['unhealthy', 'less than healthy', 'no healthy', 'less healthy']: return 'Unhealthy'
                else: return 'Unknown'
            X_out['Dietary Habits'] = X_out['Dietary Habits'].apply(clean_diet)

        # Gộp các cột Profession & Degree → Occupation
        if 'Profession' in X_out.columns and 'Degree' in X_out.columns:
            X_out['Occupation'] = X_out['Profession'].fillna(X_out['Degree'])
            
        # Gộp Work & Academic Pressure
        if 'Work Pressure' in X_out.columns and 'Academic Pressure' in X_out.columns:
            X_out['Pressure'] = X_out['Work Pressure'].fillna(X_out['Academic Pressure'])

        # Gộp Job & Study Satisfaction
        if 'Job Satisfaction' in X_out.columns and 'Study Satisfaction' in X_out.columns:
            X_out['Satisfaction'] = X_out['Job Satisfaction'].fillna(X_out['Study Satisfaction'])
            
        return X_out

class RareLabelEncoder(BaseEstimator, TransformerMixin):
    """Nhóm các nhãn hiếm (< threshold) thành 'Other'"""
    def __init__(self, variables=None, threshold=5):
        self.variables = variables or []
        self.threshold = threshold
        self.valid_labels_ = {} 

    def fit(self, X, y=None):
        for col in self.variables:
            if col in X.columns:
                counts = X[col].value_counts()
                self.valid_labels_[col] = counts[counts > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X_out = X.copy()
        for col in self.variables:
            if col in X_out.columns:
                valid_list = self.valid_labels_.get(col, [])
                X_out[col] = np.where(X_out[col].isin(valid_list), X_out[col], 'Other')
        return X_out

class ScoreBasedSelector(BaseEstimator, TransformerMixin):
    """Chọn feature dựa trên Mutual Information"""
    def __init__(self, threshold=0.01): 
        self.threshold = threshold
        self.selected_features_ = []
    def fit(self, X, y):
        print(f"\n--- [SELECTOR] Calculating Mutual Information... ---")
        X_temp = X.copy()
        cat_cols = X_temp.select_dtypes(include=['object', 'category']).columns
        num_cols = X_temp.select_dtypes(exclude=['object', 'category']).columns
        X_temp[num_cols] = X_temp[num_cols].fillna(0)
        ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_temp[cat_cols] = ord_enc.fit_transform(X_temp[cat_cols].fillna('Missing'))
        mi_scores = mutual_info_classif(X_temp, y, discrete_features='auto', random_state=42)
        score_df = pd.DataFrame({'feature': X_temp.columns, 'score': mi_scores}).sort_values(by='score', ascending=False)
        print(score_df)
        self.selected_features_ = score_df[score_df['score'] > self.threshold]['feature'].tolist()
        print(f"Selected {len(self.selected_features_)} features.")
        return self
    def transform(self, X):
        return X[self.selected_features_]


In [6]:
df_train = pd.read_csv('train.csv').drop_duplicates()
target_col = 'Depression'
for c in ['id', 'Name', 'PassengerId']:
    if c in df_train.columns: df_train.drop(c, axis=1, inplace=True)

X = df_train.drop(target_col, axis=1)
y = df_train[target_col]

# Split Train / Val / Test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()


Train: (98486, 17), Val: (21109, 17), Test: (21105, 17)


In [7]:
vars_to_rare = ['Occupation', 'Degree', 'Profession', 'City']

# Pipeline làm sạch & nhóm nhãn hiếm
pre_cleaner = Pipeline([
    ('cleaner', LogicalCleaner()), 
    ('rare_encoder', RareLabelEncoder(variables=vars_to_rare, threshold=5))
])
# Fit trên Train
X_train_pre = pre_cleaner.fit_transform(X_train, y_train)
X_val_pre = pre_cleaner.transform(X_val)
X_test_pre = pre_cleaner.transform(X_test)
# Feature Selection dựa trên MI
selector = ScoreBasedSelector(threshold=0.01)
selector.fit(X_train_pre, y_train)

final_features = selector.selected_features_
X_train_selected = X_train_pre[final_features]
X_val_selected = X_val_pre[final_features]
X_test_selected = X_test_pre[final_features]


--- [SELECTOR] Calculating Mutual Information... ---
                                  feature     score
1                                     Age  0.199349
5                       Academic Pressure  0.139114
17                             Occupation  0.138555
4                              Profession  0.138386
6                           Work Pressure  0.136494
3         Working Professional or Student  0.130896
9                        Job Satisfaction  0.130431
7                                    CGPA  0.116988
8                      Study Satisfaction  0.115846
13  Have you ever had suicidal thoughts ?  0.076460
12                                 Degree  0.041100
18                               Pressure  0.040142
15                       Financial Stress  0.030054
14                       Work/Study Hours  0.023215
11                         Dietary Habits  0.016373
0                                  Gender  0.013199
19                           Satisfaction  0.012718
16       F

In [8]:
# Phân loại numeric & categorical
cat_cols = X_train_selected.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train_selected.select_dtypes(exclude=['object', 'category']).columns.tolist()

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Other')),
    ('target_enc', TargetEncoder(smooth='auto', random_state=42)),
    ('scaler', StandardScaler())
])

final_preprocessor = ColumnTransformer(
    transformers=[('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)],
    verbose_feature_names_out=False
)

# Fit trên Train, Transform Val & Test
X_train_processed = final_preprocessor.fit_transform(X_train_selected, y_train)
X_val_processed = final_preprocessor.transform(X_val_selected)
X_test_processed = final_preprocessor.transform(X_test_selected)


In [9]:
xgb_model = xgb.XGBClassifier(
    n_estimators=3000, learning_rate=0.01, max_depth=5,
    scale_pos_weight=scale_pos_weight, eval_metric='aucpr',
    early_stopping_rounds=50, random_state=42, n_jobs=-1
)

xgb_model.fit(
    X_train_processed, y_train,
    eval_set=[(X_train_processed, y_train), (X_val_processed, y_val)],
    verbose=100
)


[0]	validation_0-aucpr:0.82433	validation_1-aucpr:0.82241
[100]	validation_0-aucpr:0.86602	validation_1-aucpr:0.86333
[200]	validation_0-aucpr:0.88596	validation_1-aucpr:0.88219
[300]	validation_0-aucpr:0.89835	validation_1-aucpr:0.89313
[400]	validation_0-aucpr:0.90530	validation_1-aucpr:0.89925
[500]	validation_0-aucpr:0.90877	validation_1-aucpr:0.90224
[600]	validation_0-aucpr:0.91090	validation_1-aucpr:0.90335
[700]	validation_0-aucpr:0.91248	validation_1-aucpr:0.90398
[800]	validation_0-aucpr:0.91355	validation_1-aucpr:0.90446
[900]	validation_0-aucpr:0.91450	validation_1-aucpr:0.90478
[1000]	validation_0-aucpr:0.91546	validation_1-aucpr:0.90497
[1100]	validation_0-aucpr:0.91646	validation_1-aucpr:0.90500
[1137]	validation_0-aucpr:0.91680	validation_1-aucpr:0.90497


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,50
,enable_categorical,False
,eval_metric,'aucpr'


In [10]:
y_val_prob = xgb_model.predict_proba(X_val_processed)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal Threshold: {best_threshold:.4f}")


Optimal Threshold: 0.7594


In [11]:
y_test_prob = xgb_model.predict_proba(X_test_processed)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int)

print(f"ROC-AUC: {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"F1: {f1_score(y_test, y_test_pred):.4f}")
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))


ROC-AUC: 0.9740
Accuracy: 0.9374
F1: 0.8306
              precision    recall  f1-score   support

           0       0.97      0.96      0.96     17270
           1       0.82      0.84      0.83      3835

    accuracy                           0.94     21105
   macro avg       0.89      0.90      0.90     21105
weighted avg       0.94      0.94      0.94     21105

[[16543   727]
 [  595  3240]]


In [12]:
print("\n>>> 8. EXPORTING SYSTEM...")
valid_labels_dict = pre_cleaner.named_steps['rare_encoder'].valid_labels_

ui_config = {
    "model_threshold": float(best_threshold),
    "input_fields": []
}

ui_features_set = set()
ui_features_set.add("Working Professional or Student")

BLACKLIST_UI = ['Occupation', 'Pressure', 'Satisfaction']

for feat in final_features:
    if feat == 'Occupation':
        ui_features_set.add('Degree')
        ui_features_set.add('Profession')
    elif feat == 'Pressure':
        ui_features_set.add('Academic Pressure')
        ui_features_set.add('Work Pressure')
    elif feat == 'Satisfaction':
        ui_features_set.add('Study Satisfaction')
        ui_features_set.add('Job Satisfaction')
    elif feat not in BLACKLIST_UI:
        ui_features_set.add(feat)

for col in ui_features_set:
    if col == "Working Professional or Student": continue
    if col not in X_train.columns: continue

    field_info = {"name": col, "label": col.replace("_", " ").title()}
    is_numeric = pd.api.types.is_numeric_dtype(X_train[col].dtype)
    is_scale_var = any(x in col for x in ['Pressure', 'Satisfaction', 'Stress'])
    
    if is_numeric and not is_scale_var:
        field_info["type"] = "number"
        field_info["min"] = float(X_train[col].min())
        field_info["max"] = float(X_train[col].max())
    else:
        field_info["type"] = "select"
        
        # Logic lấy options
        if col in ['Sleep Duration', 'Dietary Habits']: 
            temp_cleaner = LogicalCleaner()
            temp_df = temp_cleaner.transform(X_train[[col]])
            raw = temp_df[col].unique().tolist()
        elif col in valid_labels_dict:
            raw = valid_labels_dict[col]
        else:
            raw = X_train[col].value_counts().head(30).index.tolist()

        options = [str(x) for x in raw if str(x) != 'nan' and str(x) != 'Unknown']
        options = sorted(options)
        
        if col in ['Degree', 'Profession']:
            if 'Other' not in options: options.append('Other')
            
        field_info["options"] = options

    ui_config["input_fields"].append(field_info)

with open('model_ui_config.json', 'w', encoding='utf-8') as f:
    json.dump(ui_config, f, indent=4, ensure_ascii=False)

# Pipeline dùng cho inference (đã bao gồm các bước xử lý)
pipeline_inference = Pipeline([
    ('cleaner', pre_cleaner.named_steps['cleaner']),
    ('rare_encoder', pre_cleaner.named_steps['rare_encoder'])
])

full_system = {
    'selector_pipeline': pipeline_inference, 
    'preprocessor': final_preprocessor,
    'model': xgb_model,
    'threshold': best_threshold,
    'required_features': final_features 
}
joblib.dump(full_system, 'depression_prediction_system.pkl')
print("\n✅ System Saved Successfully.")


>>> 8. EXPORTING SYSTEM...

✅ System Saved Successfully.


In [2]:
import pandas as pd
try:
    df = pd.read_csv('train.csv') # Thay tên file của bạn nếu khác
except FileNotFoundError:
    print("Không tìm thấy file dữ liệu.")
    exit()
cat_cols = df.select_dtypes(include=['object', 'category']).columns
print(f"\n{'='*20} DANH SÁCH GIÁ TRỊ UNIQUE {'='*20}\n")

for col in cat_cols:
    unique_values = df[col].unique()
    num_unique = len(unique_values)
    if num_unique > 50: 
        print(f"Cột [{col}] có quá nhiều giá trị ({num_unique}). Bỏ qua hiển thị.")
        print("-" * 60)
        continue

    print(f"Cột: [{col}] - Có {num_unique} giá trị khác nhau:")
    print(unique_values)
    
    print("-" * 60)


==================== DANH SÁCH GIÁ TRỊ UNIQUE ====================

Cột [Name] có quá nhiều giá trị (422). Bỏ qua hiển thị.
------------------------------------------------------------
Cột: [Gender] - Có 2 giá trị khác nhau:
['Female' 'Male']
------------------------------------------------------------
Cột [City] có quá nhiều giá trị (98). Bỏ qua hiển thị.
------------------------------------------------------------
Cột: [Working Professional or Student] - Có 2 giá trị khác nhau:
['Working Professional' 'Student']
------------------------------------------------------------
Cột [Profession] có quá nhiều giá trị (65). Bỏ qua hiển thị.
------------------------------------------------------------
Cột: [Sleep Duration] - Có 36 giá trị khác nhau:
['More than 8 hours' 'Less than 5 hours' '5-6 hours' '7-8 hours'
 'Sleep_Duration' '1-2 hours' '6-8 hours' '4-6 hours' '6-7 hours'
 '10-11 hours' '8-9 hours' '40-45 hours' '9-11 hours' '2-3 hours'
 '3-4 hours' 'Moderate' '55-66 hours' '4-5 hours' 